In [ ]:
!pip  install tensorflow 
!pip  install boto3

In [ ]:
import boto3
import tensorflow as tf
from collections import Counter
import re
import io

# 1. 配置 RustFS 连接信息 (S3 兼容)
s3_client = boto3.client(
    's3',
    endpoint_url='http://rust-work.yy.com', # 替换为你的 RustFS 地址
    aws_access_key_id='rustfsadmin',
    aws_secret_access_key='rustfsadmin'
)

bucket_name = 'b001' # 假设 bucket 名称为 rustfs
object_key = 'monitor.txt'
output_key = 'word_count_result002.txt'

def process_and_save():
    # 列出所有 bucket，确认 b001 是否存在
    response = s3_client.list_buckets()
    print("Buckets:", [b['Name'] for b in response['Buckets']])

    # 2. 从 RustFS 下载文件内容
    response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    text_content = response['Body'].read().decode('utf-8')
    
    # 3. 使用 TensorFlow 进行处理 (或直接使用 Python 处理)
    # 这里演示如何将数据传入 TensorFlow Tensor 进行计算
    tensor_text = tf.constant(text_content)
    
    # 词频统计
    words = re.findall(r'\w+', tensor_text.numpy().decode('utf-8').lower())
    word_counts = Counter(words)
    
    # 格式化结果
    result_str = "\n".join([f"{word}: {count}" for word, count in word_counts.items()])
    
    # 4. 将结果写回 RustFS
    s3_client.put_object(
        Bucket=bucket_name,
        Key=output_key,
        Body=result_str
    )
    print(f"成功处理并保存至: s3://{bucket_name}/{output_key}")

# 执行
process_and_save()

Buckets: ['b001', 'bucket-test01-by-mc']
成功处理并保存至: s3://b001/word_count_result.txt
